## Scenario Simulation Results

### Objective
Evaluate how pricing and rate adjustments impact lock volume and revenue under a controlled simulation.

This serves as a baseline experiment before introducing statistical or machine learning models.

---

### Scenarios
- **baseline**: No change  
- **price_12.5 / price_25**: Price reductions (bps)  
- **rate_12.5 / rate_25**: Rate reductions  

---

### Approach
- Apply predefined adjustments to pricing or rate  
- Estimate resulting changes in:
  - Lock rate  
  - Lock volume  
  - Revenue  

---

### Key Observations

- **Price adjustments**
  - Increase lock volume moderately  
  - Reduce margins → **negative revenue impact**

- **Rate adjustments**
  - Strong increase in lock volume  
  - Volume gain outweighs margin loss → **positive revenue impact**

---

### Highlights

- `price_25`  
  - Locks ↑ (~155K)  
  - Revenue ↓ (~$126M)

- `rate_25`  
  - Locks ↑ (~380K)  
  - Revenue ↑ (~$907M)

---

### Takeaway
Rate-based adjustments appear more effective than price cuts for driving both volume and total revenue in this simulated environment.

---

### Notes
- This is a **scenario-based experiment**, not a predictive model  
- Assumes simplified borrower response to pricing changes  
- Results are directional and used to guide further modeling

In [26]:
import numpy as np
import pandas as pd
pd.options.display.float_format = '{:,.3f}'.format

In [27]:
df = pd.read_parquet("mortgage_data.parquet", engine="pyarrow")

In [28]:
def compute_lock_prob(df, new_rate_diff):

    # borrowre starts slightly lower than 50% tendency to lock
    logit = -0.50 

    # if new rate_diff is negative, quote is better than market, negative becomes positive
    # lock probablity goes up; better pricing relative to market increases lock probability
    logit += -5.0 * new_rate_diff

    # purchase helps most, refinance a little, cashout hurts
    logit += np.where(df["purpose"] == "purchase", 0.45, 0)
    logit += np.where(df["purpose"] == "refinance", 0.10, 0)
    logit += np.where(df["purpose"] == "cash_out", -0.25, 0)

    # primary helps, second home hurts a little, investment hurts most
    logit += np.where(df["occupancy"] == "primary", 0.15, 0)
    logit += np.where(df["occupancy"] == "second_home", -0.10, 0)
    logit += np.where(df["occupancy"] == "investment", -0.30, 0)

    # higher fico helps but moderately
    logit += 0.003 * (df["fico"] - 700)

    # higher ltv hurts, lower ltv helps
    logit += -0.015 * (df["ltv"] - 80)

    # large loans reduce lock probability
    logit += -0.000001 * (df["loan_amount"] - 450000)

    # market volatility, low volatility helps, high hurts
    logit += np.where(df["market_volatility"] == "low", 0.20, 0)
    logit += np.where(df["market_volatility"] == "medium", 0.00, 0)
    logit += np.where(df["market_volatility"] == "high", -0.35, 0)
    
    # if loan is refinance, pricig isbetter than market, give boost
    logit += np.where((df["purpose"] == "refinance") & (new_rate_diff < 0), 0.35, 0)
    
    # converts logit into a probability between 0 and 1
    lock_prob = 1 / (1 + np.exp(-logit))

    # prevents extreme values
    lock_prob = np.clip(lock_prob, 0.02, 0.98)
    
    return lock_prob

In [29]:
scenarios = {
    "baseline": 0.000,
    "rate_improve_12.5": -0.125,
    "rate_improve_25": -0.250,
}

results = []

# on average we earn 125 bps per loan
base_margin_bps = 125

# Loop through each pricing/rate scenario
for name, shift in scenarios.items():

    # Apply the scenario shift to the original quote-vs-market difference
    # Negative shift = more competitive pricing / better rate for borrower
    new_rate_diff = df["rate_diff"] + shift

    # Estimate lock probability for each loan under this scenario
    prob = compute_lock_prob(df, new_rate_diff)

    # Reduce our margin by the size of the pricing concession
    # Example: -0.125 shift means giving up 12.5 bps of margin
    margin_bps = base_margin_bps - abs(shift * 100)

    # Expected revenue per loan =
    # loan amount × margin % × probability of lock
    expected_revenue = (
        df["loan_amount"] * (margin_bps / 10000) * prob
    )

    # Store scenario-level summary results
    results.append({
        "scenario": name,                     # scenario label
        "avg_lock_rate": round(prob.mean(), 4),         # average expected lock rate
        "expected_locks": prob.sum(),         # total expected locks
        "avg_margin_bps": margin_bps,         # assumed margin in bps
        "total_revenue": expected_revenue.sum()  # total expected revenue
    })

# Convert results list into a DataFrame for easy comparison
results_df = pd.DataFrame(results)

baseline = results_df.loc[results_df["scenario"] == "baseline"].iloc[0]

results_df["delta_locks"] = results_df["expected_locks"] - baseline["expected_locks"]
results_df["delta_lock_rate"] = results_df["avg_lock_rate"] - baseline["avg_lock_rate"]
results_df["delta_revenue"] = results_df["total_revenue"] - baseline["total_revenue"]

In [30]:
def compute_lock_prob(df, new_rate_diff, price_improve_bps=0):
    logit = -0.50

    # Rate competitiveness effect
    logit += -5.0 * new_rate_diff

    # Price improvement effect
    # positive price_improve_bps = better pricing for borrower
    logit += 0.02 * price_improve_bps

    # Loan purpose effects
    logit += np.where(df["purpose"] == "purchase", 0.45, 0)
    logit += np.where(df["purpose"] == "refinance", 0.10, 0)
    logit += np.where(df["purpose"] == "cash_out", -0.25, 0)

    # Occupancy effects
    logit += np.where(df["occupancy"] == "primary", 0.15, 0)
    logit += np.where(df["occupancy"] == "second_home", -0.10, 0)
    logit += np.where(df["occupancy"] == "investment", -0.30, 0)

    # Credit / leverage
    logit += 0.003 * (df["fico"] - 700)
    logit += -0.015 * (df["ltv"] - 80)
    logit += -0.000001 * (df["loan_amount"] - 450000)

    # Market environment
    logit += np.where(df["market_volatility"] == "low", 0.20, 0)
    logit += np.where(df["market_volatility"] == "medium", 0.00, 0)
    logit += np.where(df["market_volatility"] == "high", -0.35, 0)

    # Refinance borrowers are extra sensitive to better rate
    logit += np.where((df["purpose"] == "refinance") & (new_rate_diff < 0), 0.35, 0)

    lock_prob = 1 / (1 + np.exp(-logit))
    lock_prob = np.clip(lock_prob, 0.02, 0.98)

    return lock_prob

In [31]:
price_scenarios = {
    "baseline": 0.0,
    "price_improve_12.5": 12.5,
    "price_improve_25": 25.0,
}

price_results = []
base_margin_bps = 125

for name, price_improve_bps in price_scenarios.items():
    # Rate stays unchanged
    new_rate_diff = df["rate_diff"]

    # Price improves, so probability can increase
    prob = compute_lock_prob(df, new_rate_diff, price_improve_bps=price_improve_bps)

    # Better price lowers our margin
    margin_bps = base_margin_bps - price_improve_bps

    expected_revenue = (
        df["loan_amount"] * (margin_bps / 10000) * prob
    )

    price_results.append({
        "scenario": name,
        "avg_lock_rate": round(prob.mean(), 4),
        "expected_locks": prob.sum(),
        "avg_margin_bps": margin_bps,
        "total_revenue": expected_revenue.sum()
    })

price_results_df = pd.DataFrame(price_results)


baseline = price_results_df.loc[price_results_df["scenario"] == "baseline"].iloc[0]
price_results_df["delta_locks"] = price_results_df["expected_locks"] - baseline["expected_locks"]
price_results_df["delta_lock_rate"] = price_results_df["avg_lock_rate"] - baseline["avg_lock_rate"]
price_results_df["delta_revenue"] = price_results_df["total_revenue"] - baseline["total_revenue"]


In [32]:
# adding type column to differentiate between 
# rate and price scenarios when we combine them for analysis

results_df["type"] = "rate"
price_results_df["type"] = "price"

results_df["scenario"] = results_df["scenario"].str.replace("rate_improve_", "rate_")
price_results_df["scenario"] = price_results_df["scenario"].str.replace("price_improve_", "price_")

In [33]:
combined_df = pd.concat([results_df, price_results_df], ignore_index=True)

combined_df = combined_df.sort_values(by=["type", "scenario"])

In [34]:
baseline = combined_df[combined_df["scenario"] == "baseline"].iloc[0]

combined_df["delta_revenue"] = combined_df["total_revenue"] - baseline["total_revenue"]
combined_df["delta_lock_rate"] = combined_df["avg_lock_rate"] - baseline["avg_lock_rate"]
combined_df["efficiency"] = combined_df["delta_revenue"] / abs(combined_df["avg_margin_bps"] - 125)

In [35]:
combined_df

,scenario,avg_lock_rate,expected_locks,avg_margin_bps,total_revenue,delta_locks,delta_lock_rate,delta_revenue,type,efficiency
3,baseline,0.506,"758,512.497",125.000,"4,164,351,537.708",0.000,0.000,0.000,price,NaN
4,price_12.5,0.558,"837,100.177",112.500,"4,147,978,215.779","78,587.680",0.052,"-16,373,321.929",price,"-1,309,865.754"
5,price_25,0.609,"914,114.689",100.000,"4,037,378,817.234","155,602.192",0.104,"-126,972,720.475",price,"-5,078,908.819"
0,baseline,0.506,"758,512.497",125.000,"4,164,351,537.708",0.000,0.000,0.000,rate,NaN
1,rate_12.5,0.644,"965,992.709",112.500,"4,808,745,994.623","207,480.212",0.138,"644,394,456.915",rate,"51,551,556.553"
2,rate_25,0.759,"1,139,167.853",100.000,"5,071,733,980.243","380,655.356",0.254,"907,382,442.534",rate,"36,295,297.701"


## Interpretation of Results

- Price reductions increase lock volume but reduce overall revenue due to margin compression  
- Rate reductions significantly increase lock volume and result in higher total revenue  

- Larger adjustments (25bps) amplify both effects:
  - `price_25` → higher volume but revenue loss  
  - `rate_25` → strong volume growth and highest revenue gain  

### Conclusion
In this simulation, rate-based adjustments outperform price cuts in driving both conversion and revenue.

These results provide a directional baseline for more advanced modeling in the next step.